In [1]:
!pip install -U langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.9
    Uninstalling langchain-1.3.9:
      Successfully uninstalled langchain-1.3.9


In [2]:
import os
from datetime import datetime

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
# API Key
os.environ["GOOGLE_API_KEY"] = api_key

In [6]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    api_key=api_key
)

# -----------------------------
# Tools
# -----------------------------

@tool
def calculator(expression: str) -> str:
    """Use this tool for arithmetic calculations."""
    try:
        allowed = "0123456789+-*/(). "
        if not all(ch in allowed for ch in expression):
            return "Only simple arithmetic is allowed."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


@tool
def get_current_time() -> str:
    """Use this tool to get the current date and time."""
    return str(datetime.now())


@tool
def simple_knowledge_search(query: str) -> str:
    """Use this tool to search a small built-in knowledge base."""
    knowledge = {
        "rag": "RAG means Retrieval Augmented Generation. It retrieves relevant documents before answering.",
        "langchain": "LangChain is a framework for building LLM applications using prompts, chains, tools, agents and memory.",
        "agent": "An AI agent can reason, use tools, remember context and perform tasks.",
        "machine learning": "Machine Learning is a branch of AI where systems learn patterns from data."
    }

    query_lower = query.lower()

    for key, value in knowledge.items():
        if key in query_lower:
            return value

    return "No matching knowledge found."


# -----------------------------
# Multi-task Agent
# -----------------------------

agent = create_agent(
    model=llm,
    tools=[
        calculator,
        get_current_time,
        simple_knowledge_search
    ],
    system_prompt="""
    You are a multi-task AI assistant.

    You can do many tasks:
    1. Answer general questions
    2. Perform calculations using calculator
    3. Tell current time using get_current_time
    4. Write blogs, emails, notes and summaries
    5. Search basic knowledge using simple_knowledge_search

    Rules:
    - Use tools when useful.
    - Answer in simple English.
    - Be clear and practical.
    """
)


In [7]:

# -----------------------------
# Chat Loop
# -----------------------------

print("Multi-task LangChain Agent Started")
print("Type 'exit' to stop\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    response = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_input
            }
        ]
    })

    print("\nAgent:")
    print(response["messages"][-1].content)
    print("-" * 60)

Multi-task LangChain Agent Started
Type 'exit' to stop

You: what is rag?

Agent:
[{'type': 'text', 'text': "RAG stands for Retrieval Augmented Generation. It's a method that retrieves relevant documents before generating an answer.", 'extras': {'signature': 'CswCAQw51sdhCiW0maUZq5p9N75QS7M3ntNOFj0JnxCFjMU5msHZY+eN2TAD5r1EqzBaGF9ZpLFkQi8wY0UkA8AWQWm7l2nq7DlxdXN6mAjBAMUch92tSArPu3buDGU+KaEyKagLpElSLSCRg1kxFBrJEMzfyQ96DIDSvaLABEWJjbr1GucFWhxO7z/HGAR7qp4R8DRdjZnOS4o1+g406wvomyAOQJYquNbGJFuuf1fg4qGD8fcak1Bopcjjx1421hj1oyxtkxAPv8qMQDshtG5wkeN5g8IWefYhiTy9De4bxz5sUemeMEMxvB9xylU1fH/qnyQfIBta7Lj7EKcO3pcjKuCD9G8uzc7OrGQESNVjn9+JTA7KeQPqnf22UVvFpP4LQXgkZfkZ2/hJqKVURO8/ULtTVHgDkEYdMO+I3QjXEcaqwMNkRlo0k1lKnqc='}}]
------------------------------------------------------------
You: Calculate 25000 * 12 + 5000

Agent:
The result of the calculation is 305000.
------------------------------------------------------------
You: What is the current time?

Agent:
The current time is 05:08:00 on June 24, 202